# 11 — Evaluating transfer to continuous areas of London

The borough-based evaluation holds out administrative groups, but the held-out boroughs in one round may be spread across London and locations on opposite sides of a borough boundary can remain very close. This notebook applies a stricter geographical test: London is divided into five continuous regions using coordinates alone, and each region is hidden in turn while the model is trained on the other four.

The same 12 representative models used in the random-versus-borough comparison are evaluated. Results are then compared across three settings: randomly mixed locations, held-out borough groups and held-out continuous regions. The notebook also examines whether held-out prediction errors remain geographically clustered.

## Main findings

Holding out a continuous part of London is substantially more demanding for PTAL than either random division or borough hold-out. The location-only PTAL model falls to mean R² = −0.129 under continuous-region testing, while the representation models retain useful predictive ability. Full fusion gives the highest mean R² (0.547), followed closely by Street View content plus coverage information (0.540), the compact street-and-aerial combination (0.524), and DINOv2 (0.450). When all held-out predictions are pooled, full fusion and Street View are effectively tied at R² ≈ 0.602.

For EPC, the change from borough to continuous-region testing is much smaller. With compact property controls, pooled R² is 0.394 for full fusion, 0.390 for the compact sky-and-space combination and 0.357 for DINOv2. The richer EPC control model remains close to R² = 0.60. TESSERA adds only a very small improvement (pooled R² 0.591 versus 0.588), while adding all representations to these rich controls reduces performance (0.571).

The two richer EPC specifications that initially reached the lower Ridge-penalty limit were rechecked with a grid extending to the unregularised endpoint. Six of the ten refined fits selected `alpha = 0`, but the largest change in any held-out R² was below 0.00000008. The earlier performance conclusions are therefore unchanged and no unresolved tuning boundary remains.

Held-out errors remain geographically patterned. Under continuous-region testing, Moran's I is 0.709 for the location-only PTAL model and falls to 0.475–0.496 for the representation models. For EPC with compact controls it falls from 0.187 to 0.040–0.054 after adding representations. The richer EPC models have lower residual clustering (0.042–0.050), although it is not eliminated. These values are diagnostic effect sizes: the permutation p-values all reach the minimum available value of 0.005 with 199 permutations and are not used to select models.

Overall, representations are especially important for transferring PTAL predictions into a geographically unseen area, whereas EPC performance is more stable across validation designs once direct building information is available.

In [ ]:
# Connect Google Drive and load packages for spatial grouping, modelling and residual analysis.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import gc
import time
import hashlib
import platform
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn
from scipy import sparse

from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith("_")
})

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

required_paths = [
    FINAL_MODEL_TABLE_PATH,
    FEATURE_MANIFEST_JSON_PATH,
    INCREMENTAL_RUN_SPEC_PATH,
    INCREMENTAL_AUDIT_PATH,
    INCREMENTAL_RESULTS_PATH,
    INCREMENTAL_PREDICTIONS_PATH,
    RANDOM_CV_RUN_SPEC_PATH,
    RANDOM_CV_AUDIT_PATH,
    RANDOM_CV_RESULTS_PATH,
    RANDOM_CV_PREDICTIONS_PATH,
    PCA64_RUN_SPEC_PATH,
    PCA64_AUDIT_PATH,
]
for path in required_paths:
    assert path.exists(), f"Missing prerequisite: {path}"

print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)
print("Required frozen inputs:", len(required_paths), "/", len(required_paths))


## 1. Load the common data and established model definitions

The common table, feature manifest, representative model set and earlier random and borough results are loaded. Stored model and data identifiers ensure that the three evaluation settings differ in geographical division rather than in predictors or samples.

In [ ]:
# Read the common data, representative models and earlier protocol results.
df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
with open(FEATURE_MANIFEST_JSON_PATH, "r") as f:
    manifest = json.load(f)
with open(INCREMENTAL_RUN_SPEC_PATH, "r") as f:
    source_07_spec = json.load(f)
with open(INCREMENTAL_AUDIT_PATH, "r") as f:
    source_07_audit = json.load(f)
with open(RANDOM_CV_RUN_SPEC_PATH, "r") as f:
    source_08_spec = json.load(f)
with open(RANDOM_CV_AUDIT_PATH, "r") as f:
    source_08_audit = json.load(f)
with open(PCA64_RUN_SPEC_PATH, "r") as f:
    source_10_spec = json.load(f)
with open(PCA64_AUDIT_PATH, "r") as f:
    source_10_audit = json.load(f)

for audit_name, audit in [
    ("07", source_07_audit),
    ("08", source_08_audit),
    ("10", source_10_audit),
]:
    assert audit["integrity_gate_pass"] is True, f"Notebook {audit_name} integrity gate did not pass"
    assert audit["interpretation_gate_pass"] is True, f"Notebook {audit_name} interpretation gate did not pass"

target_col = manifest["target_column"]
group_col = manifest["group_column"]
coordinate_cols = list(manifest["coordinate_columns"])
categorical_master = set(manifest["categorical_columns"])
feature_sets = manifest["feature_sets"]

assert coordinate_cols[:2] == ["x", "y"]
assert len(df) == 26597
assert df["sample_id"].is_unique
assert df[target_col].notna().all()
assert df[group_col].notna().all()
assert df[group_col].nunique() == 33
assert set(df["task"].unique()) == {"PTAL", "EPC"}
assert np.isfinite(df[["x", "y"]].to_numpy(dtype=float)).all()

df = df.sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
task_counts = df["task"].value_counts().to_dict()
assert task_counts == {"EPC": 20000, "PTAL": 6597}

model_key_frame = df[["sample_id", "task", group_col, target_col]].copy()
model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(model_key_frame, index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(
    json.dumps(manifest, sort_keys=True).encode("utf-8")
).hexdigest()
coordinate_frame = df[["sample_id", "task", "x", "y"]].copy()
coordinate_hash = hashlib.sha256(
    pd.util.hash_pandas_object(coordinate_frame, index=False).values.tobytes()
).hexdigest()

def json_sha256(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode("utf-8")).hexdigest()

source_07_run_spec_sha256 = json_sha256(source_07_spec)
source_08_run_spec_sha256 = json_sha256(source_08_spec)
source_10_run_spec_sha256 = json_sha256(source_10_spec)

assert source_07_spec["model_key_sha256"] == model_key_hash
assert source_08_spec["model_key_sha256"] == model_key_hash
assert source_10_spec["model_key_sha256"] == model_key_hash
assert source_07_spec["feature_manifest_sha256"] == manifest_hash
assert source_08_spec["feature_manifest_sha256"] == manifest_hash
assert source_10_spec["feature_manifest_sha256"] == manifest_hash
assert source_07_audit["run_spec_sha256"] == source_07_run_spec_sha256
assert source_08_audit["run_spec_sha256"] == source_08_run_spec_sha256
assert source_10_audit["run_spec_sha256"] == source_10_run_spec_sha256
assert int(source_07_spec["outer_splits"]) == int(RIDGE_OUTER_SPLITS) == 5
assert int(source_07_spec["inner_splits"]) == int(RIDGE_INNER_SPLITS) == 3
assert [float(x) for x in source_07_spec["alpha_grid"]] == [float(x) for x in RIDGE_ALPHA_GRID]

print("Frozen prerequisite chain: PASS")
print("Model-key SHA256:", model_key_hash)
print("Feature-manifest SHA256:", manifest_hash)
print("Coordinate SHA256:", coordinate_hash)
display(pd.Series(task_counts, name="n"))


## 2. Reconstruct the 12 representative models

The set contains PTAL and EPC control baselines, DINOv2, selected compact combinations, full fusion and the richer EPC reference models. Using the same role-based subset as Notebook 08 keeps the spatial comparison focused and avoids choosing models in response to the continuous-region results.

In [ ]:
# Reconstruct the same 12 representative model specifications.
selected_definitions = [
    {"task": "PTAL", "base_feature_set": "PTAL_spatial_baseline", "added_feature_set": None, "analysis_role": "control_baseline"},
    {"task": "PTAL", "base_feature_set": "PTAL_spatial_baseline", "added_feature_set": "DINOv2", "analysis_role": "aerial_single"},
    {"task": "PTAL", "base_feature_set": "PTAL_spatial_baseline", "added_feature_set": "StreetView_CLIP_plus_metadata", "analysis_role": "streetview_content_plus_coverage"},
    {"task": "PTAL", "base_feature_set": "PTAL_spatial_baseline", "added_feature_set": "Street_and_sky", "analysis_role": "compact_fusion"},
    {"task": "PTAL", "base_feature_set": "PTAL_spatial_baseline", "added_feature_set": "All_representations_plus_SV_metadata", "analysis_role": "full_fusion"},
    {"task": "EPC", "base_feature_set": "EPC_controls_sparse", "added_feature_set": None, "analysis_role": "sparse_control_baseline"},
    {"task": "EPC", "base_feature_set": "EPC_controls_sparse", "added_feature_set": "DINOv2", "analysis_role": "aerial_single"},
    {"task": "EPC", "base_feature_set": "EPC_controls_sparse", "added_feature_set": "Sky_and_space", "analysis_role": "compact_fusion"},
    {"task": "EPC", "base_feature_set": "EPC_controls_sparse", "added_feature_set": "All_representations_plus_SV_metadata", "analysis_role": "full_fusion"},
    {"task": "EPC", "base_feature_set": "EPC_controls_extensive", "added_feature_set": None, "analysis_role": "privileged_control_baseline"},
    {"task": "EPC", "base_feature_set": "EPC_controls_extensive", "added_feature_set": "TESSERA", "analysis_role": "small_positive_representation_check"},
    {"task": "EPC", "base_feature_set": "EPC_controls_extensive", "added_feature_set": "All_representations_plus_SV_metadata", "analysis_role": "full_fusion_redundancy_check"},
]

def dedupe_preserve_order(columns):
    return list(dict.fromkeys(columns))

def columns_sha256(columns):
    return hashlib.sha256(
        json.dumps(list(columns), separators=(",", ":")).encode("utf-8")
    ).hexdigest()

source_08_models = {
    row["model_id"]: row for row in source_08_spec["selected_model_specifications"]
}
source_07_models = {
    row["model_id"]: row for row in source_07_spec["model_specifications"]
}
selected_records = []
selected_model_features = {}

for definition in selected_definitions:
    task = definition["task"]
    base = definition["base_feature_set"]
    added = definition["added_feature_set"]
    assert base in feature_sets
    if added is None:
        model_id = base
        columns = list(feature_sets[base])
    else:
        assert added in feature_sets
        model_id = f"{base}__plus__{added}"
        columns = dedupe_preserve_order(feature_sets[base] + feature_sets[added])

    assert model_id in source_08_models and model_id in source_07_models
    frozen_08 = source_08_models[model_id]
    frozen_07 = source_07_models[model_id]
    assert frozen_08["task"] == task == frozen_07["task"]
    assert frozen_08["analysis_role"] == definition["analysis_role"]
    assert int(frozen_08["n_features_manifest"]) == len(columns)
    assert int(frozen_07["n_features_manifest"]) == len(columns)
    assert frozen_08["feature_columns_sha256"] == columns_sha256(columns)
    assert frozen_07["feature_columns_sha256"] == columns_sha256(columns)
    assert not [c for c in columns if c not in df.columns]

    selected_model_features[model_id] = columns
    selected_records.append({
        **definition,
        "model_id": model_id,
        "n_features_manifest": len(columns),
        "feature_columns_sha256": columns_sha256(columns),
    })

selected_specs = pd.DataFrame(selected_records)
assert len(selected_specs) == 12
assert selected_specs["model_id"].is_unique
assert selected_specs.groupby("task").size().to_dict() == {"EPC": 7, "PTAL": 5}

display(selected_specs)
print("Exact Notebook-08 model subset verified:", len(selected_specs))


## 3. Retain the same Street View treatment

Locations without Street View imagery keep missing visual vectors for training-based filling. Only known structural values, such as zero contributing images, are set directly. This maintains comparability with the earlier models.

In [ ]:
# Retain the established treatment of Street View non-coverage.
SV_META_COLS = [
    "sv_has_streetview", "sv_n_images", "sv_min_dist_m", "sv_mean_dist_m"
]
SV_CLIP_COLS = feature_sets["StreetView_CLIP_only"]

def apply_structural_sv_metadata_fill(task_df, task):
    out = task_df.copy()
    radius = {"PTAL": float(STREETVIEW_PTAL_RADIUS_M), "EPC": float(STREETVIEW_EPC_RADIUS_M)}[task]
    has = pd.to_numeric(out["sv_has_streetview"], errors="coerce")
    clip_complete = out[SV_CLIP_COLS].notna().all(axis=1)
    clip_all_missing = out[SV_CLIP_COLS].isna().all(axis=1)
    has = has.where(has.notna(), clip_complete.astype(int)).astype(int)
    assert has.isin([0, 1]).all()
    assert clip_complete[has.eq(1)].all()
    assert clip_all_missing[has.eq(0)].all()

    out["sv_has_streetview"] = has
    for c in SV_META_COLS[1:]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    no_sv = has.eq(0)
    out.loc[no_sv, "sv_n_images"] = 0.0
    for c in ["sv_min_dist_m", "sv_mean_dist_m"]:
        out.loc[no_sv & out[c].isna(), c] = radius
    assert out.loc[no_sv, "sv_n_images"].eq(0).all()
    assert out.loc[no_sv, ["sv_min_dist_m", "sv_mean_dist_m"]].notna().all().all()
    return out

sv_audit = []
for task in ["PTAL", "EPC"]:
    filled = apply_structural_sv_metadata_fill(df[df["task"] == task], task)
    has = filled["sv_has_streetview"].eq(1)
    sv_audit.append({
        "task": task,
        "n": int(len(filled)),
        "n_with_streetview": int(has.sum()),
        "coverage_pct": float(100 * has.mean()),
        "n_no_streetview": int((~has).sum()),
    })
display(pd.DataFrame(sv_audit))


## 4. Retain the same Ridge training pipeline

Numerical and categorical preparation and Ridge penalty selection are learned from the current training regions only. A single processing job is used at a time to limit memory use while fitting the wide combined representations.

In [ ]:
# Build the same training-only preprocessing and Ridge pipeline.
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_pipeline(feature_cols):
    categorical_cols = [c for c in feature_cols if c in categorical_master]
    numeric_cols = [c for c in feature_cols if c not in categorical_master]
    transformers = []
    if numeric_cols:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ))
    if categorical_cols:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                ("onehot", make_onehot()),
            ]),
            categorical_cols,
        ))
    pre = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.0)
    return Pipeline([
        ("preprocess", pre),
        ("ridge", Ridge(solver="lsqr", max_iter=5000, tol=1e-4)),
    ])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    assert mapped.notna().all()
    return mapped.astype(bool)


## 5. Create five continuous geographical regions

K-means clustering uses only British National Grid `x` and `y` coordinates to divide the common sample into five compact areas. The same partition is used for PTAL and EPC. Region labels are ordered from west to east for readable tables and maps, and the assignment is stored so every model uses identical held-out areas.

In [ ]:
# Create five continuous regions using coordinates only and order them west to east.
coords = df[["x", "y"]].to_numpy(dtype=float)
kmeans = KMeans(
    n_clusters=RIDGE_OUTER_SPLITS,
    random_state=RANDOM_STATE,
    n_init=50,
    algorithm="lloyd",
)
raw_labels = kmeans.fit_predict(coords)
raw_centres = kmeans.cluster_centers_
canonical_order = sorted(
    range(RIDGE_OUTER_SPLITS),
    key=lambda label: (float(raw_centres[label, 0]), float(raw_centres[label, 1])),
)
raw_to_canonical = {raw: canonical for canonical, raw in enumerate(canonical_order)}
geometric_labels = np.array([raw_to_canonical[int(x)] for x in raw_labels], dtype=int)

centres_rows = []
for raw_label in canonical_order:
    canonical_label = raw_to_canonical[raw_label]
    centres_rows.append({
        "geometric_outer_fold": int(canonical_label),
        "centre_x": float(raw_centres[raw_label, 0]),
        "centre_y": float(raw_centres[raw_label, 1]),
    })
geometric_centres = pd.DataFrame(centres_rows).sort_values("geometric_outer_fold")

geometric_folds = df[["sample_id", "task", group_col, target_col, "x", "y"]].copy()
geometric_folds["geometric_outer_fold"] = geometric_labels
geometric_folds = geometric_folds.rename(columns={target_col: "target"})
geometric_folds = geometric_folds[[
    "sample_id", "task", "geometric_outer_fold", group_col, "target", "x", "y"
]].sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)

qa_rows = []
for (task, fold), g in geometric_folds.groupby(["task", "geometric_outer_fold"]):
    qa_rows.append({
        "task": task,
        "geometric_outer_fold": int(fold),
        "n": int(len(g)),
        "share_task": float(len(g) / task_counts[task]),
        "n_boroughs": int(g[group_col].nunique()),
        "x_min": float(g["x"].min()),
        "x_max": float(g["x"].max()),
        "y_min": float(g["y"].min()),
        "y_max": float(g["y"].max()),
        "mean_x": float(g["x"].mean()),
        "mean_y": float(g["y"].mean()),
        "target_mean": float(g["target"].mean()),
        "target_sd": float(g["target"].std(ddof=1)),
    })
geometric_qa = pd.DataFrame(qa_rows).sort_values(["task", "geometric_outer_fold"])

assert len(geometric_folds) == len(df)
assert geometric_folds["sample_id"].is_unique
assert set(geometric_folds["geometric_outer_fold"]) == set(range(RIDGE_OUTER_SPLITS))
assert geometric_qa.groupby("task").size().eq(RIDGE_OUTER_SPLITS).all()
assert geometric_qa["n"].ge(100).all()
assert geometric_qa["share_task"].ge(0.05).all(), "A geometric fold has <5% of a task"

if GEOMETRIC_OUTER_FOLDS_PATH.exists():
    existing = pd.read_csv(GEOMETRIC_OUTER_FOLDS_PATH).sort_values(
        ["task", "sample_id"], kind="mergesort"
    ).reset_index(drop=True)
    pd.testing.assert_frame_equal(
        existing[geometric_folds.columns],
        geometric_folds,
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-9,
    )
    print("Verified existing frozen geometric assignments.")
else:
    atomic_csv(geometric_folds, GEOMETRIC_OUTER_FOLDS_PATH)

atomic_csv(geometric_qa, GEOMETRIC_BLOCK_QA_PATH)

assignment_key = geometric_folds[["sample_id", "task", "geometric_outer_fold"]]
geometric_fold_hash = hashlib.sha256(
    pd.util.hash_pandas_object(assignment_key, index=False).values.tobytes()
).hexdigest()

selected_model_payload = [
    {
        "task": row.task,
        "model_id": row.model_id,
        "base_feature_set": row.base_feature_set,
        "added_feature_set": row.added_feature_set if pd.notna(row.added_feature_set) else None,
        "analysis_role": row.analysis_role,
        "n_features_manifest": int(row.n_features_manifest),
        "feature_columns_sha256": row.feature_columns_sha256,
    }
    for row in selected_specs.itertuples(index=False)
]

run_spec = {
    "run_spec_version": "11-v1-2026-08-23",
    "notebook": "11_geometric_spatial_blocks_and_residual_diagnostics.ipynb",
    "purpose": "target-independent compact-region transfer sensitivity and OOF residual spatial diagnosis",
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "coordinate_sha256": coordinate_hash,
    "geometric_outer_fold_assignment_sha256": geometric_fold_hash,
    "source_07_run_spec_sha256": source_07_run_spec_sha256,
    "source_08_run_spec_sha256": source_08_run_spec_sha256,
    "source_10_run_spec_sha256": source_10_run_spec_sha256,
    "sklearn_version": sklearn.__version__,
    "geometric_splitter": {
        "class": "KMeans coordinate regions + leave-one-region-out",
        "fit_population": "all 26,597 common-sample rows; coordinates only; one shared partition for both tasks",
        "coordinate_crs": "British National Grid coordinates inherited from canonical table",
        "n_regions": int(RIDGE_OUTER_SPLITS),
        "random_state": int(RANDOM_STATE),
        "n_init": 50,
        "algorithm": "lloyd",
        "canonical_label_rule": "cluster centres sorted by x then y",
        "canonical_centres": geometric_centres.to_dict("records"),
    },
    "inner_splitter": {
        "class": "GroupKFold",
        "n_splits": int(RIDGE_INNER_SPLITS),
        "groups": "remaining geometric region labels",
    },
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "inner_selection_metric": "RMSE",
    "selected_model_specifications": selected_model_payload,
    "protocol_comparators": ["Notebook 08 random nested CV", "Notebook 07 borough-grouped nested CV"],
    "protocol_inference": "descriptive; unlike fold definitions are not paired and receive no fold-level p-values",
    "residual_diagnostic": {
        "graph": "symmetrised 8-nearest-neighbour graph, row-standardised",
        "coordinate_columns": ["x", "y"],
        "statistic": "global Moran's I on OOF residual = observed - predicted",
        "permutations": 199,
        "p_value": "two-sided around randomisation expectation; deterministic model-specific seed",
    },
    "dinov3_status": "out of remaining dissertation scope",
}
run_spec_sha256 = json_sha256(run_spec)

if GEOMETRIC_RUN_SPEC_PATH.exists():
    with open(GEOMETRIC_RUN_SPEC_PATH, "r") as f:
        existing_spec = json.load(f)
    assert existing_spec == run_spec, (
        "Existing Notebook-11 checkpoints belong to another run specification. "
        "Do not mix outputs; archive them before a documented rerun."
    )
else:
    atomic_json(run_spec, GEOMETRIC_RUN_SPEC_PATH)

expected_keys = {
    (row.task, row.model_id, fold)
    for row in selected_specs.itertuples(index=False)
    for fold in range(RIDGE_OUTER_SPLITS)
}
expected_prediction_rows = int(sum(
    task_counts[row.task] for row in selected_specs.itertuples(index=False)
))
assert len(expected_keys) == 60
assert expected_prediction_rows == 172985

display(geometric_centres)
display(geometric_qa)
print("Geometric-fold SHA256:", geometric_fold_hash)
print("Notebook-11 run-spec SHA256:", run_spec_sha256)
print("Expected geometric runs:", len(expected_keys))
print("Expected geometric predictions:", expected_prediction_rows)


## 6. Fit models while holding out each region

One complete region forms the outer test set in each round. The Ridge penalty is chosen inside the remaining four regions using region-aware divisions, so no part of the held-out area influences model preparation. Per-region predictions and metrics are saved after each fit.

In [ ]:
# Fit each model while holding out one complete geographical region.
if GEOMETRIC_CV_RESULTS_PATH.exists():
    completed = pd.read_csv(GEOMETRIC_CV_RESULTS_PATH)
    required = {"task", "model_id", "geometric_outer_fold", "run_spec_sha256"}
    assert required.issubset(completed.columns)
    assert not completed.duplicated(["task", "model_id", "geometric_outer_fold"]).any()
    assert completed["run_spec_sha256"].eq(run_spec_sha256).all()
    completed["geometric_outer_fold"] = completed["geometric_outer_fold"].astype(int)
    completed_keys = set(map(
        tuple,
        completed[["task", "model_id", "geometric_outer_fold"]].to_numpy(),
    ))
    assert completed_keys.issubset(expected_keys)
else:
    completed = pd.DataFrame()

result_rows = [] if completed.empty else completed.to_dict("records")

def checkpoint_is_valid(path, task, model_id, fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {
            "sample_id", "task", "model_id", "geometric_outer_fold",
            "borough_code", "x", "y", "y_true", "y_pred", "run_spec_sha256",
        }
        return (
            required.issubset(p.columns)
            and p["sample_id"].is_unique
            and p["task"].eq(task).all()
            and p["model_id"].eq(model_id).all()
            and p["geometric_outer_fold"].astype(int).eq(fold).all()
            and p["run_spec_sha256"].eq(run_spec_sha256).all()
            and np.isfinite(p[["x", "y", "y_true", "y_pred"]].to_numpy(dtype=float)).all()
            and set(p["sample_id"].astype(str)) == set(pd.Series(expected_ids).astype(str))
        )
    except Exception:
        return False

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    task_df = apply_structural_sv_metadata_fill(task_df, task)
    task_assignment = geometric_folds[geometric_folds["task"] == task].set_index("sample_id")
    task_assignment = task_assignment.loc[task_df["sample_id"]].reset_index()
    assert task_assignment["sample_id"].equals(task_df["sample_id"])
    assert np.allclose(task_assignment["target"], task_df[target_col], rtol=0, atol=1e-12)

    y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
    boroughs = task_df[group_col].astype(str).to_numpy()
    blocks = task_assignment["geometric_outer_fold"].to_numpy(dtype=int)
    task_specs = selected_specs[selected_specs["task"] == task]

    for spec_row in task_specs.itertuples(index=False):
        model_id = spec_row.model_id
        cols = selected_model_features[model_id]
        X = task_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(blocks == outer_fold)
            train_idx = np.flatnonzero(blocks != outer_fold)
            run_key = (task, model_id, outer_fold)
            pred_file = GEOMETRIC_CV_CHUNK_DIR / f"geometric__{task}__{model_id}__fold{outer_fold}.parquet"
            completed_keys_now = {
                (r["task"], r["model_id"], int(r["geometric_outer_fold"]))
                for r in result_rows
            }
            if run_key in completed_keys_now and checkpoint_is_valid(
                pred_file, task, model_id, outer_fold, task_df.iloc[test_idx]["sample_id"]
            ):
                print("SKIP validated checkpoint:", run_key)
                continue

            print("\n" + "=" * 100)
            print("GEOMETRIC CV |", task, "|", model_id, "| held-out region", outer_fold)
            print("=" * 100)

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            inner_groups = blocks[train_idx]
            assert set(inner_groups) == (set(range(RIDGE_OUTER_SPLITS)) - {outer_fold})
            assert set(blocks[train_idx]).isdisjoint(set(blocks[test_idx]))

            inner_splits = list(GroupKFold(RIDGE_INNER_SPLITS).split(X_train, y_train, inner_groups))
            for inner_train, inner_valid in inner_splits:
                assert set(inner_groups[inner_train]).isdisjoint(set(inner_groups[inner_valid]))

            search = GridSearchCV(
                estimator=build_pipeline(cols),
                param_grid={"ridge__alpha": RIDGE_ALPHA_GRID},
                scoring="neg_root_mean_squared_error",
                cv=inner_splits,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise",
            )
            t0 = time.time()
            search.fit(X_train, y_train)
            elapsed_s = time.time() - t0
            pred = search.predict(X_test)
            assert len(pred) == len(test_idx)
            assert np.isfinite(pred).all()

            best_alpha = float(search.best_params_["ridge__alpha"])
            train_boroughs = set(boroughs[train_idx])
            test_boroughs = set(boroughs[test_idx])
            row = {
                "task": task,
                "model_id": model_id,
                "analysis_role": spec_row.analysis_role,
                "geometric_outer_fold": int(outer_fold),
                "validation_scheme": "geometric_region_nested_5fold",
                "n_features_manifest": int(len(cols)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "n_train_geometric_blocks": int(len(np.unique(blocks[train_idx]))),
                "n_test_geometric_blocks": int(len(np.unique(blocks[test_idx]))),
                "n_train_boroughs": int(len(train_boroughs)),
                "n_test_boroughs": int(len(test_boroughs)),
                "n_boroughs_in_both": int(len(train_boroughs.intersection(test_boroughs))),
                "best_alpha": best_alpha,
                "alpha_grid_edge": bool(best_alpha in {float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))}),
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, pred)),
                "rmse": rmse(y_test, pred),
                "mae": float(mean_absolute_error(y_test, pred)),
                "fit_seconds": float(elapsed_s),
                "run_spec_sha256": run_spec_sha256,
            }
            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task,
                "model_id": model_id,
                "geometric_outer_fold": int(outer_fold),
                "borough_code": boroughs[test_idx],
                "x": task_df.iloc[test_idx]["x"].to_numpy(dtype=float),
                "y": task_df.iloc[test_idx]["y"].to_numpy(dtype=float),
                "y_true": y_test,
                "y_pred": pred,
                "run_spec_sha256": run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)

            result_rows = [
                r for r in result_rows
                if (r["task"], r["model_id"], int(r["geometric_outer_fold"])) != run_key
            ]
            result_rows.append(row)
            atomic_csv(
                pd.DataFrame(result_rows).sort_values(["task", "model_id", "geometric_outer_fold"]),
                GEOMETRIC_CV_RESULTS_PATH,
            )
            print(row)
            del search, pred, pred_frame, X_train, X_test
            gc.collect()

print("Notebook-11 geometric fitting complete or safely checkpointed.")


## 7. Assemble predictions for all three evaluation settings

The continuous-region analysis requires 60 model rounds and 172,985 held-out predictions. Each prediction file is matched to the intended sample IDs, coordinates, outcome, region and model definition. Random and borough predictions are then read for the same 12 models and samples.

In [ ]:
# Verify continuous-region predictions and load matching random and borough results.
geometric_results = pd.read_csv(GEOMETRIC_CV_RESULTS_PATH)
geometric_results["geometric_outer_fold"] = geometric_results["geometric_outer_fold"].astype(int)
assert not geometric_results.duplicated(["task", "model_id", "geometric_outer_fold"]).any()
assert geometric_results["run_spec_sha256"].eq(run_spec_sha256).all()
actual_keys = set(map(
    tuple,
    geometric_results[["task", "model_id", "geometric_outer_fold"]].to_numpy(),
))
assert actual_keys == expected_keys, "Notebook 11 incomplete: rerun Section 6"

geometric_pred_frames = []
chunk_audit_rows = []
for task, model_id, outer_fold in sorted(expected_keys):
    path = GEOMETRIC_CV_CHUNK_DIR / f"geometric__{task}__{model_id}__fold{outer_fold}.parquet"
    assert path.exists(), f"Missing geometric prediction chunk: {path.name}"
    p = pd.read_parquet(path)
    required = {
        "sample_id", "task", "model_id", "geometric_outer_fold", "borough_code",
        "x", "y", "y_true", "y_pred", "run_spec_sha256",
    }
    assert required.issubset(p.columns)
    assert p["sample_id"].is_unique
    assert p["task"].eq(task).all() and p["model_id"].eq(model_id).all()
    assert p["geometric_outer_fold"].astype(int).eq(outer_fold).all()
    assert p["run_spec_sha256"].eq(run_spec_sha256).all()
    assert np.isfinite(p[["x", "y", "y_true", "y_pred"]].to_numpy(dtype=float)).all()

    expected = geometric_folds[
        (geometric_folds["task"] == task)
        & (geometric_folds["geometric_outer_fold"] == outer_fold)
    ][["sample_id", "borough_code", "target", "x", "y"]].sort_values("sample_id").reset_index(drop=True)
    observed = p[["sample_id", "borough_code", "y_true", "x", "y"]].sort_values("sample_id").reset_index(drop=True)
    assert expected["sample_id"].equals(observed["sample_id"])
    assert expected["borough_code"].astype(str).equals(observed["borough_code"].astype(str))
    assert np.allclose(expected["target"], observed["y_true"], rtol=0, atol=1e-12)
    assert np.allclose(expected[["x", "y"]], observed[["x", "y"]], rtol=0, atol=1e-9)

    chunk_audit_rows.append({
        "task": task,
        "model_id": model_id,
        "geometric_outer_fold": int(outer_fold),
        "n_rows": int(len(p)),
        "file": path.name,
    })
    geometric_pred_frames.append(p)

geometric_preds = pd.concat(geometric_pred_frames, ignore_index=True)
assert len(geometric_preds) == expected_prediction_rows
assert not geometric_preds.duplicated(["sample_id", "task", "model_id"]).any()

selected_ids = set(selected_specs["model_id"])

borough_results = pd.read_csv(INCREMENTAL_RESULTS_PATH)
borough_results = borough_results[borough_results["model_id"].isin(selected_ids)].copy()
borough_results["outer_fold"] = borough_results["outer_fold"].astype(int)
assert len(borough_results) == len(expected_keys)
assert borough_results["run_spec_sha256"].eq(source_07_run_spec_sha256).all()

borough_preds = pd.read_parquet(INCREMENTAL_PREDICTIONS_PATH)
borough_preds = borough_preds[borough_preds["model_id"].isin(selected_ids)].copy()
assert len(borough_preds) == expected_prediction_rows
assert borough_preds["run_spec_sha256"].eq(source_07_run_spec_sha256).all()
assert not borough_preds.duplicated(["sample_id", "task", "model_id"]).any()

random_results = pd.read_csv(RANDOM_CV_RESULTS_PATH)
random_results = random_results[random_results["model_id"].isin(selected_ids)].copy()
random_results["random_outer_fold"] = random_results["random_outer_fold"].astype(int)
assert len(random_results) == len(expected_keys)
assert random_results["run_spec_sha256"].eq(source_08_run_spec_sha256).all()

random_preds = pd.read_parquet(RANDOM_CV_PREDICTIONS_PATH)
random_preds = random_preds[random_preds["model_id"].isin(selected_ids)].copy()
assert len(random_preds) == expected_prediction_rows
assert random_preds["run_spec_sha256"].eq(source_08_run_spec_sha256).all()
assert not random_preds.duplicated(["sample_id", "task", "model_id"]).any()

for row in selected_specs.itertuples(index=False):
    n_expected = task_counts[row.task]
    for p in [random_preds, borough_preds, geometric_preds]:
        assert len(p[(p["task"] == row.task) & (p["model_id"] == row.model_id)]) == n_expected

print("Validated geometric runs:", len(actual_keys))
print("Validated geometric chunks:", len(chunk_audit_rows))
print("Validated geometric prediction rows:", len(geometric_preds))
print("Validated comparator prediction rows:", len(random_preds), len(borough_preds))


## 8. Compare random, borough and continuous-region performance

The three settings use different partitions, so round numbers are not paired. Mean round metrics describe variation across held-out groups, while pooled metrics combine every prediction made while a sample was hidden. Pooled results are particularly informative when continuous regions differ in size or outcome variance.

In [ ]:
# Summarise performance under all three geographical evaluation settings.
def summarise_scheme(fold_results, predictions, scheme, fold_col):
    rows = []
    for spec in selected_specs.itertuples(index=False):
        g = fold_results[(fold_results["task"] == spec.task) & (fold_results["model_id"] == spec.model_id)]
        p = predictions[(predictions["task"] == spec.task) & (predictions["model_id"] == spec.model_id)]
        assert len(g) == RIDGE_OUTER_SPLITS and g[fold_col].nunique() == RIDGE_OUTER_SPLITS
        assert len(p) == task_counts[spec.task]
        rows.append({
            "task": spec.task,
            "model_id": spec.model_id,
            "analysis_role": spec.analysis_role,
            "base_feature_set": spec.base_feature_set,
            "added_feature_set": spec.added_feature_set,
            "n_features": int(spec.n_features_manifest),
            "validation_scheme": scheme,
            "mean_r2": float(g["r2"].mean()),
            "sd_r2": float(g["r2"].std(ddof=1)),
            "mean_rmse": float(g["rmse"].mean()),
            "sd_rmse": float(g["rmse"].std(ddof=1)),
            "mean_mae": float(g["mae"].mean()),
            "sd_mae": float(g["mae"].std(ddof=1)),
            "pooled_r2": float(r2_score(p["y_true"], p["y_pred"])),
            "pooled_rmse": rmse(p["y_true"], p["y_pred"]),
            "pooled_mae": float(mean_absolute_error(p["y_true"], p["y_pred"])),
            "alpha_edge_hits": int(coerce_bool(g["alpha_grid_edge"]).sum()),
            "median_best_alpha": float(g["best_alpha"].median()),
            "total_fit_minutes": float(g["fit_seconds"].sum() / 60),
        })
    return pd.DataFrame(rows)

random_summary = summarise_scheme(random_results, random_preds, "random_nested_5fold", "random_outer_fold")
borough_summary = summarise_scheme(borough_results, borough_preds, "borough_grouped_nested_5fold", "outer_fold")
geometric_summary = summarise_scheme(
    geometric_results, geometric_preds, "geometric_region_nested_5fold", "geometric_outer_fold"
)
protocol_summary = pd.concat([random_summary, borough_summary, geometric_summary], ignore_index=True)
assert len(protocol_summary) == 36

identity = ["task", "model_id", "analysis_role", "base_feature_set", "added_feature_set", "n_features"]
metric_cols = [
    "mean_r2", "sd_r2", "mean_rmse", "sd_rmse", "mean_mae", "sd_mae",
    "pooled_r2", "pooled_rmse", "pooled_mae", "alpha_edge_hits", "median_best_alpha", "total_fit_minutes",
]

def suffix_metrics(frame, suffix):
    return frame[identity + metric_cols].rename(columns={c: f"{c}_{suffix}" for c in metric_cols})

comparison = suffix_metrics(random_summary, "random").merge(
    suffix_metrics(borough_summary, "borough"), on=identity, validate="one_to_one"
).merge(
    suffix_metrics(geometric_summary, "geometric"), on=identity, validate="one_to_one"
)
assert len(comparison) == 12

for metric in ["mean_r2", "pooled_r2"]:
    comparison[f"random_minus_borough_{metric}"] = comparison[f"{metric}_random"] - comparison[f"{metric}_borough"]
    comparison[f"random_minus_geometric_{metric}"] = comparison[f"{metric}_random"] - comparison[f"{metric}_geometric"]
    comparison[f"borough_minus_geometric_{metric}"] = comparison[f"{metric}_borough"] - comparison[f"{metric}_geometric"]
for metric in ["mean_rmse", "pooled_rmse", "mean_mae", "pooled_mae"]:
    comparison[f"borough_minus_random_{metric}"] = comparison[f"{metric}_borough"] - comparison[f"{metric}_random"]
    comparison[f"geometric_minus_random_{metric}"] = comparison[f"{metric}_geometric"] - comparison[f"{metric}_random"]
    comparison[f"geometric_minus_borough_{metric}"] = comparison[f"{metric}_geometric"] - comparison[f"{metric}_borough"]

display(protocol_summary.sort_values(["task", "model_id", "validation_scheme"]))
display(comparison[[
    "task", "model_id", "analysis_role",
    "pooled_r2_random", "pooled_r2_borough", "pooled_r2_geometric",
    "random_minus_borough_pooled_r2", "borough_minus_geometric_pooled_r2",
    "pooled_rmse_random", "pooled_rmse_borough", "pooled_rmse_geometric",
]])


## 9. Examine residual geography and finalise the EPC penalty choice

The first continuous-region run selected the smallest candidate Ridge penalty (`alpha = 0.001`) in three of five regions for two low-dimensional EPC models: the richer property-control model and the same model with TESSERA. This pattern indicates that these specifications are approaching an unregularised linear fit; it does not invalidate the other results.

To establish the appropriate endpoint without repeating the complete experiment, this section refits only those two models across the same five regions using an extended grid that includes the natural lower limit, `alpha = 0`. The original 60 runs remain intact, the ten refined fits are stored separately, and the refined rows are used in the final comparison tables. The section then calculates Moran's I from held-out residuals to show whether nearby locations tend to be over- or under-predicted together.

In [ ]:
# Refine the two near-unregularised EPC fits, then calculate residual spatial clustering.
def build_knn_weights(task_coordinates, k=8):
    n = len(task_coordinates)
    assert n > k
    nn = NearestNeighbors(n_neighbors=k + 1, algorithm="auto", metric="euclidean")
    neighbours = nn.fit(task_coordinates).kneighbors(
        task_coordinates, return_distance=False
    )
    # Querying the training coordinates includes each observation itself.
    # Remove the exact row index rather than assuming it appears first because
    # duplicated coordinates can produce tied distances.
    neighbour_rows = []
    for i, candidate_indices in enumerate(neighbours):
        without_self = candidate_indices[candidate_indices != i]
        assert len(without_self) >= k
        neighbour_rows.append(without_self[:k])
    rows = np.repeat(np.arange(n), k)
    cols = np.asarray(neighbour_rows, dtype=int).reshape(-1)
    data = np.ones(len(rows), dtype=float)
    weights = sparse.csr_matrix((data, (rows, cols)), shape=(n, n))
    weights = weights.maximum(weights.T).tocsr()
    weights.setdiag(0)
    weights.eliminate_zeros()
    row_sums = np.asarray(weights.sum(axis=1)).ravel()
    assert np.all(row_sums > 0)
    return (sparse.diags(1.0 / row_sums) @ weights).tocsr()


def moran_with_permutations(residuals, weights, seed, permutations=199):
    centred = np.asarray(residuals, dtype=float)
    centred = centred - centred.mean()
    n = len(centred)
    denominator = float(centred @ centred)
    assert denominator > 0
    weight_sum = float(weights.sum())
    observed = float(
        (n / weight_sum) * (centred @ (weights @ centred)) / denominator
    )
    expected = float(-1.0 / (n - 1))

    rng = np.random.default_rng(seed)
    permuted = np.empty((n, permutations), dtype=float)
    for j in range(permutations):
        permuted[:, j] = rng.permutation(centred)
    weighted = weights @ permuted
    numerators = np.sum(permuted * weighted, axis=0)
    denominators = np.sum(permuted * permuted, axis=0)
    permuted_i = (n / weight_sum) * numerators / denominators
    p_two_sided = float(
        (1 + np.sum(np.abs(permuted_i - expected) >= abs(observed - expected)))
        / (permutations + 1)
    )
    return observed, expected, p_two_sided


# Identify repeated lower-grid selections in the completed 60-run ledger.
base_geometric_results = geometric_results.copy()
base_geometric_preds = geometric_preds.copy()
base_edge_rows = base_geometric_results[
    coerce_bool(base_geometric_results["alpha_grid_edge"])
].copy()
base_edge_counts = (
    base_edge_rows.groupby(["task", "model_id"])
    .size().reset_index(name="edge_hits")
)
repeated_base_edges = base_edge_counts[base_edge_counts["edge_hits"] >= 3].copy()

refinement_model_keys = sorted(
    map(tuple, repeated_base_edges[["task", "model_id"]].to_numpy())
)
refinement_grid = sorted(set(
    [0.0, 1e-6, 1e-5, 1e-4]
    + [float(value) for value in RIDGE_ALPHA_GRID]
))

REFINEMENT_RESULTS_PATH = (
    FINAL_MODEL_DIR / "11_geometric_alpha_refinement_fold_results.csv"
)
REFINEMENT_COMPARISON_PATH = (
    FINAL_MODEL_DIR / "11_geometric_alpha_refinement_comparison.csv"
)
REFINEMENT_CHUNK_DIR = FINAL_MODEL_DIR / "11_geometric_alpha_refinement_chunks"
REFINEMENT_CHUNK_DIR.mkdir(parents=True, exist_ok=True)
REFINEMENT_SPEC_PATH = AUDIT_DIR / "11_geometric_alpha_refinement_spec.json"
FINAL_REFINED_RESULTS_PATH = (
    FINAL_MODEL_DIR / "11_geometric_cv_fold_results_refined.csv"
)

if refinement_model_keys:
    repeated_edge_rows = base_edge_rows.merge(
        repeated_base_edges[["task", "model_id"]],
        on=["task", "model_id"], how="inner", validate="many_to_one",
    )
    # The observed issue is only at the lower edge. An upper-edge selection
    # would require a different expansion and is not silently handled here.
    assert repeated_edge_rows["best_alpha"].eq(float(min(RIDGE_ALPHA_GRID))).all(), (
        "A repeated upper alpha boundary was found; inspect it separately."
    )

    refinement_spec = {
        "purpose": "Resolve repeated lower-alpha selections for geometric-region models only",
        "source_geometric_run_spec_sha256": run_spec_sha256,
        "model_key_sha256": model_key_hash,
        "feature_manifest_sha256": manifest_hash,
        "geometric_outer_fold_assignment_sha256": geometric_fold_hash,
        "trigger_rule": "at least 3 of 5 folds selected the original minimum alpha",
        "refinement_models": [
            {"task": task, "model_id": model_id}
            for task, model_id in refinement_model_keys
        ],
        "original_alpha_grid": [float(value) for value in RIDGE_ALPHA_GRID],
        "refinement_alpha_grid": refinement_grid,
        "lower_endpoint_interpretation": (
            "alpha=0 is the natural unregularised endpoint and is not treated "
            "as evidence that a still-smaller value is required"
        ),
        "outer_splits": int(RIDGE_OUTER_SPLITS),
        "inner_splits": int(RIDGE_INNER_SPLITS),
        "inner_grouping": "remaining geometric regions",
        "scoring": "neg_root_mean_squared_error",
        "random_state": int(RANDOM_STATE),
    }
    refinement_spec_sha256 = json_sha256(refinement_spec)
    atomic_json(refinement_spec, REFINEMENT_SPEC_PATH)

    expected_refinement_keys = {
        (task, model_id, fold)
        for task, model_id in refinement_model_keys
        for fold in range(RIDGE_OUTER_SPLITS)
    }

    if REFINEMENT_RESULTS_PATH.exists():
        refinement_results = pd.read_csv(REFINEMENT_RESULTS_PATH)
        required_refinement_cols = {
            "task", "model_id", "geometric_outer_fold",
            "alpha_refinement_spec_sha256",
        }
        assert required_refinement_cols.issubset(refinement_results.columns)
        assert not refinement_results.duplicated(
            ["task", "model_id", "geometric_outer_fold"]
        ).any()
        assert refinement_results["alpha_refinement_spec_sha256"].eq(
            refinement_spec_sha256
        ).all()
        refinement_results["geometric_outer_fold"] = (
            refinement_results["geometric_outer_fold"].astype(int)
        )
        existing_refinement_keys = set(map(
            tuple,
            refinement_results[
                ["task", "model_id", "geometric_outer_fold"]
            ].to_numpy(),
        ))
        assert existing_refinement_keys.issubset(expected_refinement_keys)
    else:
        refinement_results = pd.DataFrame()

    refinement_rows = (
        [] if refinement_results.empty
        else refinement_results.to_dict("records")
    )

    def refinement_checkpoint_is_valid(path, task, model_id, fold, expected_ids):
        if not path.exists():
            return False
        try:
            pred = pd.read_parquet(path)
            required = {
                "sample_id", "task", "model_id", "geometric_outer_fold",
                "borough_code", "x", "y", "y_true", "y_pred",
                "run_spec_sha256", "alpha_refinement_spec_sha256",
            }
            return (
                required.issubset(pred.columns)
                and pred["sample_id"].is_unique
                and pred["task"].eq(task).all()
                and pred["model_id"].eq(model_id).all()
                and pred["geometric_outer_fold"].astype(int).eq(fold).all()
                and pred["run_spec_sha256"].eq(run_spec_sha256).all()
                and pred["alpha_refinement_spec_sha256"].eq(
                    refinement_spec_sha256
                ).all()
                and np.isfinite(
                    pred[["x", "y", "y_true", "y_pred"]].to_numpy(dtype=float)
                ).all()
                and set(pred["sample_id"].astype(str))
                == set(pd.Series(expected_ids).astype(str))
            )
        except Exception:
            return False

    for task, model_id in refinement_model_keys:
        task_df = df[df["task"] == task].reset_index(drop=True)
        task_df = apply_structural_sv_metadata_fill(task_df, task)
        task_assignment = geometric_folds[
            geometric_folds["task"] == task
        ].set_index("sample_id")
        task_assignment = task_assignment.loc[task_df["sample_id"]].reset_index()
        assert task_assignment["sample_id"].equals(task_df["sample_id"])

        y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
        boroughs = task_df[group_col].astype(str).to_numpy()
        blocks = task_assignment["geometric_outer_fold"].to_numpy(dtype=int)
        cols = selected_model_features[model_id]
        model_spec = selected_specs[
            (selected_specs["task"] == task)
            & (selected_specs["model_id"] == model_id)
        ].iloc[0]
        X = task_df[cols]

        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(blocks == outer_fold)
            train_idx = np.flatnonzero(blocks != outer_fold)
            refinement_key = (task, model_id, outer_fold)
            pred_file = REFINEMENT_CHUNK_DIR / (
                f"refined__{task}__{model_id}__fold{outer_fold}.parquet"
            )
            completed_refinement_keys = {
                (row["task"], row["model_id"], int(row["geometric_outer_fold"]))
                for row in refinement_rows
            }
            if (
                refinement_key in completed_refinement_keys
                and refinement_checkpoint_is_valid(
                    pred_file, task, model_id, outer_fold,
                    task_df.iloc[test_idx]["sample_id"],
                )
            ):
                print("SKIP validated alpha refinement:", refinement_key)
                continue

            print("\nALPHA REFINEMENT |", task, "|", model_id,
                  "| held-out region", outer_fold)
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            inner_groups = blocks[train_idx]
            inner_splits = list(GroupKFold(RIDGE_INNER_SPLITS).split(
                X_train, y_train, inner_groups
            ))
            for inner_train, inner_valid in inner_splits:
                assert set(inner_groups[inner_train]).isdisjoint(
                    set(inner_groups[inner_valid])
                )

            search = GridSearchCV(
                estimator=build_pipeline(cols),
                param_grid={"ridge__alpha": refinement_grid},
                scoring="neg_root_mean_squared_error",
                cv=inner_splits,
                refit=True,
                n_jobs=1,
                return_train_score=False,
                error_score="raise",
            )
            start = time.time()
            search.fit(X_train, y_train)
            elapsed_s = time.time() - start
            prediction = search.predict(X_test)
            assert len(prediction) == len(test_idx)
            assert np.isfinite(prediction).all()

            best_alpha = float(search.best_params_["ridge__alpha"])
            row = {
                "task": task,
                "model_id": model_id,
                "analysis_role": model_spec["analysis_role"],
                "geometric_outer_fold": int(outer_fold),
                "validation_scheme": "geometric_region_nested_5fold",
                "n_features_manifest": int(len(cols)),
                "n_train": int(len(train_idx)),
                "n_test": int(len(test_idx)),
                "n_train_geometric_blocks": int(len(np.unique(blocks[train_idx]))),
                "n_test_geometric_blocks": int(len(np.unique(blocks[test_idx]))),
                "n_train_boroughs": int(len(set(boroughs[train_idx]))),
                "n_test_boroughs": int(len(set(boroughs[test_idx]))),
                "n_boroughs_in_both": int(len(
                    set(boroughs[train_idx]).intersection(set(boroughs[test_idx]))
                )),
                "best_alpha": best_alpha,
                # Zero is the natural lower endpoint; only an unresolved upper
                # endpoint remains an open grid-boundary problem after refinement.
                "alpha_grid_edge": bool(best_alpha == max(refinement_grid)),
                "best_alpha_is_zero": bool(best_alpha == 0.0),
                "alpha_refinement_applied": True,
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, prediction)),
                "rmse": rmse(y_test, prediction),
                "mae": float(mean_absolute_error(y_test, prediction)),
                "fit_seconds": float(elapsed_s),
                "run_spec_sha256": run_spec_sha256,
                "alpha_refinement_spec_sha256": refinement_spec_sha256,
            }
            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task,
                "model_id": model_id,
                "geometric_outer_fold": int(outer_fold),
                "borough_code": boroughs[test_idx],
                "x": task_df.iloc[test_idx]["x"].to_numpy(dtype=float),
                "y": task_df.iloc[test_idx]["y"].to_numpy(dtype=float),
                "y_true": y_test,
                "y_pred": prediction,
                "run_spec_sha256": run_spec_sha256,
                "alpha_refinement_spec_sha256": refinement_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)

            refinement_rows = [
                existing for existing in refinement_rows
                if (
                    existing["task"], existing["model_id"],
                    int(existing["geometric_outer_fold"])
                ) != refinement_key
            ]
            refinement_rows.append(row)
            atomic_csv(
                pd.DataFrame(refinement_rows).sort_values(
                    ["task", "model_id", "geometric_outer_fold"]
                ),
                REFINEMENT_RESULTS_PATH,
            )
            print(row)
            del search, prediction, pred_frame, X_train, X_test
            gc.collect()

    refinement_results = pd.read_csv(REFINEMENT_RESULTS_PATH)
    refinement_results["geometric_outer_fold"] = (
        refinement_results["geometric_outer_fold"].astype(int)
    )
    actual_refinement_keys = set(map(
        tuple,
        refinement_results[
            ["task", "model_id", "geometric_outer_fold"]
        ].to_numpy(),
    ))
    assert actual_refinement_keys == expected_refinement_keys
    assert refinement_results["alpha_refinement_spec_sha256"].eq(
        refinement_spec_sha256
    ).all()

    refined_prediction_frames = []
    for task, model_id, fold in sorted(expected_refinement_keys):
        pred_file = REFINEMENT_CHUNK_DIR / (
            f"refined__{task}__{model_id}__fold{fold}.parquet"
        )
        expected_ids = geometric_folds[
            (geometric_folds["task"] == task)
            & (geometric_folds["geometric_outer_fold"] == fold)
        ]["sample_id"]
        assert refinement_checkpoint_is_valid(
            pred_file, task, model_id, fold, expected_ids
        )
        refined_prediction_frames.append(pd.read_parquet(pred_file))
    refined_predictions = pd.concat(refined_prediction_frames, ignore_index=True)
    assert len(refined_predictions) == sum(
        task_counts[task] for task, _ in refinement_model_keys
    )

    base_for_comparison = base_geometric_results.merge(
        repeated_base_edges[["task", "model_id"]],
        on=["task", "model_id"], how="inner", validate="many_to_one",
    )
    refinement_comparison = base_for_comparison.merge(
        refinement_results,
        on=["task", "model_id", "geometric_outer_fold"],
        suffixes=("_original", "_refined"),
        validate="one_to_one",
    )
    for metric in ["r2", "rmse", "mae", "inner_best_rmse"]:
        refinement_comparison[f"delta_{metric}_refined_minus_original"] = (
            refinement_comparison[f"{metric}_refined"]
            - refinement_comparison[f"{metric}_original"]
        )
    atomic_csv(refinement_comparison, REFINEMENT_COMPARISON_PATH)

    geometric_results = pd.concat([
        base_geometric_results.merge(
            repeated_base_edges[["task", "model_id"]].assign(_replace=True),
            on=["task", "model_id"], how="left",
        ).query("_replace != True").drop(columns="_replace"),
        refinement_results,
    ], ignore_index=True, sort=False)
    geometric_preds = pd.concat([
        base_geometric_preds.merge(
            repeated_base_edges[["task", "model_id"]].assign(_replace=True),
            on=["task", "model_id"], how="left",
        ).query("_replace != True").drop(columns="_replace"),
        refined_predictions,
    ], ignore_index=True, sort=False)
else:
    refinement_spec_sha256 = None
    refinement_results = pd.DataFrame()
    refinement_comparison = pd.DataFrame()
    geometric_results = base_geometric_results.copy()
    geometric_preds = base_geometric_preds.copy()

geometric_results["geometric_outer_fold"] = (
    geometric_results["geometric_outer_fold"].astype(int)
)
assert len(geometric_results) == len(expected_keys) == 60
assert not geometric_results.duplicated(
    ["task", "model_id", "geometric_outer_fold"]
).any()
assert len(geometric_preds) == expected_prediction_rows == 172985
assert not geometric_preds.duplicated(["task", "model_id", "sample_id"]).any()


# Rebuild the three-protocol summaries using the refined rows where applicable.
geometric_summary = summarise_scheme(
    geometric_results, geometric_preds,
    "geometric_region_nested_5fold", "geometric_outer_fold",
)
protocol_summary = pd.concat(
    [random_summary, borough_summary, geometric_summary], ignore_index=True
)
assert len(protocol_summary) == 36

identity = [
    "task", "model_id", "analysis_role", "base_feature_set",
    "added_feature_set", "n_features",
]
metric_cols = [
    "mean_r2", "sd_r2", "mean_rmse", "sd_rmse", "mean_mae", "sd_mae",
    "pooled_r2", "pooled_rmse", "pooled_mae", "alpha_edge_hits",
    "median_best_alpha", "total_fit_minutes",
]
comparison = suffix_metrics(random_summary, "random").merge(
    suffix_metrics(borough_summary, "borough"),
    on=identity, validate="one_to_one",
).merge(
    suffix_metrics(geometric_summary, "geometric"),
    on=identity, validate="one_to_one",
)
assert len(comparison) == 12
for metric in ["mean_r2", "pooled_r2"]:
    comparison[f"random_minus_borough_{metric}"] = (
        comparison[f"{metric}_random"] - comparison[f"{metric}_borough"]
    )
    comparison[f"random_minus_geometric_{metric}"] = (
        comparison[f"{metric}_random"] - comparison[f"{metric}_geometric"]
    )
    comparison[f"borough_minus_geometric_{metric}"] = (
        comparison[f"{metric}_borough"] - comparison[f"{metric}_geometric"]
    )
for metric in ["mean_rmse", "pooled_rmse", "mean_mae", "pooled_mae"]:
    comparison[f"borough_minus_random_{metric}"] = (
        comparison[f"{metric}_borough"] - comparison[f"{metric}_random"]
    )
    comparison[f"geometric_minus_random_{metric}"] = (
        comparison[f"{metric}_geometric"] - comparison[f"{metric}_random"]
    )
    comparison[f"geometric_minus_borough_{metric}"] = (
        comparison[f"{metric}_geometric"] - comparison[f"{metric}_borough"]
    )


# Calculate residual spatial clustering from held-out predictions only.
scheme_inputs = [
    ("random_nested_5fold", random_preds, "random_outer_fold"),
    ("borough_grouped_nested_5fold", borough_preds, "outer_fold"),
    ("geometric_region_nested_5fold", geometric_preds, "geometric_outer_fold"),
]
moran_rows = []
residual_fold_rows = []
for task in ["PTAL", "EPC"]:
    base = df[df["task"] == task][
        ["sample_id", "x", "y", target_col]
    ].copy()
    base = base.rename(columns={target_col: "target"})
    base = base.sort_values("sample_id").reset_index(drop=True)
    weights = build_knn_weights(
        base[["x", "y"]].to_numpy(dtype=float), k=8
    )

    for spec in selected_specs[
        selected_specs["task"] == task
    ].itertuples(index=False):
        for scheme, predictions, fold_col in scheme_inputs:
            pred = predictions[
                (predictions["task"] == task)
                & (predictions["model_id"] == spec.model_id)
            ][["sample_id", fold_col, "y_true", "y_pred"]].copy()
            merged = base.merge(
                pred, on="sample_id", how="inner", validate="one_to_one"
            )
            assert len(merged) == task_counts[task]
            assert np.allclose(
                merged["target"], merged["y_true"], rtol=0, atol=1e-12
            )
            residual = (
                merged["y_true"].to_numpy(dtype=float)
                - merged["y_pred"].to_numpy(dtype=float)
            )

            seed_text = (
                f"{RANDOM_STATE}|{task}|{spec.model_id}|{scheme}|moran199"
            )
            seed = int(
                hashlib.sha256(seed_text.encode("utf-8")).hexdigest()[:8], 16
            )
            observed_i, expected_i, p_value = moran_with_permutations(
                residual, weights, seed=seed, permutations=199
            )
            moran_rows.append({
                "task": task,
                "model_id": spec.model_id,
                "analysis_role": spec.analysis_role,
                "validation_scheme": scheme,
                "n": int(len(merged)),
                "knn_k": 8,
                "moran_i": observed_i,
                "randomisation_expected_i": expected_i,
                "permutations": 199,
                "permutation_p_two_sided": p_value,
                "residual_mean": float(residual.mean()),
                "residual_sd": float(residual.std(ddof=1)),
            })

            merged["residual"] = residual
            for fold, group in merged.groupby(fold_col):
                residual_fold_rows.append({
                    "task": task,
                    "model_id": spec.model_id,
                    "analysis_role": spec.analysis_role,
                    "validation_scheme": scheme,
                    "validation_fold": int(fold),
                    "n": int(len(group)),
                    "mean_residual_bias": float(group["residual"].mean()),
                    "rmse": rmse(group["y_true"], group["y_pred"]),
                    "mae": float(mean_absolute_error(
                        group["y_true"], group["y_pred"]
                    )),
                    "observed_mean": float(group["y_true"].mean()),
                    "predicted_mean": float(group["y_pred"].mean()),
                })

moran_summary = pd.DataFrame(moran_rows).sort_values(
    ["task", "model_id", "validation_scheme"]
)
residual_fold_summary = pd.DataFrame(residual_fold_rows).sort_values(
    ["task", "model_id", "validation_scheme", "validation_fold"]
)
assert len(moran_summary) == 36
assert len(residual_fold_summary) == 180
assert np.isfinite(
    moran_summary[["moran_i", "permutation_p_two_sided"]].to_numpy()
).all()

final_edge_counts = (
    geometric_results.assign(
        edge=coerce_bool(geometric_results["alpha_grid_edge"])
    ).groupby(["task", "model_id"])["edge"]
    .sum().reset_index(name="edge_hits")
)
unresolved_edges = final_edge_counts[final_edge_counts["edge_hits"] >= 3]

integrity_gate_pass = bool(
    actual_keys == expected_keys
    and len(base_geometric_preds) == expected_prediction_rows
    and len(chunk_audit_rows) == len(expected_keys)
    and len(geometric_results) == len(expected_keys)
    and len(geometric_preds) == expected_prediction_rows
    and len(protocol_summary) == 36
    and len(comparison) == 12
    and len(moran_summary) == 36
    and len(residual_fold_summary) == 180
    and (
        not refinement_model_keys
        or len(refinement_results) == len(refinement_model_keys) * RIDGE_OUTER_SPLITS
    )
    and source_07_audit["integrity_gate_pass"]
    and source_08_audit["integrity_gate_pass"]
    and source_10_audit["integrity_gate_pass"]
)
interpretation_gate_pass = bool(
    integrity_gate_pass
    and unresolved_edges.empty
    and source_07_audit["interpretation_gate_pass"]
    and source_08_audit["interpretation_gate_pass"]
    and source_10_audit["interpretation_gate_pass"]
)

audit = {
    # Retain the original keys used by the frozen pipeline interface.
    "run_spec_path": str(GEOMETRIC_RUN_SPEC_PATH),
    "run_spec_sha256": run_spec_sha256,
    "source_07_run_spec_sha256": source_07_run_spec_sha256,
    "source_08_run_spec_sha256": source_08_run_spec_sha256,
    "source_10_run_spec_sha256": source_10_run_spec_sha256,
    "expected_geometric_fold_runs": int(len(expected_keys)),
    "completed_geometric_fold_runs": int(len(actual_keys)),
    "expected_geometric_prediction_rows": int(expected_prediction_rows),
    "actual_geometric_prediction_rows": int(len(geometric_preds)),
    "geometric_prediction_chunks_validated": int(len(chunk_audit_rows)),
    "base_run_spec_path": str(GEOMETRIC_RUN_SPEC_PATH),
    "base_run_spec_sha256": run_spec_sha256,
    "alpha_refinement_spec_path": (
        str(REFINEMENT_SPEC_PATH) if refinement_model_keys else None
    ),
    "alpha_refinement_spec_sha256": refinement_spec_sha256,
    "base_fold_results_path": str(GEOMETRIC_CV_RESULTS_PATH),
    "final_refined_fold_results_path": str(FINAL_REFINED_RESULTS_PATH),
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "coordinate_sha256": coordinate_hash,
    "geometric_outer_fold_assignment_sha256": geometric_fold_hash,
    "selected_models": int(len(selected_specs)),
    "geometric_regions": int(RIDGE_OUTER_SPLITS),
    "completed_base_geometric_fold_runs": int(len(actual_keys)),
    "completed_alpha_refinement_runs": int(len(refinement_results)),
    "models_receiving_alpha_refinement": [
        {"task": task, "model_id": model_id}
        for task, model_id in refinement_model_keys
    ],
    "refinement_alpha_grid": refinement_grid if refinement_model_keys else None,
    "refinement_zero_alpha_selections": int(
        refinement_results.get(
            "best_alpha_is_zero", pd.Series(dtype=bool)
        ).fillna(False).astype(bool).sum()
    ),
    "refinement_max_abs_fold_delta_r2": (
        float(refinement_comparison[
            "delta_r2_refined_minus_original"
        ].abs().max())
        if len(refinement_comparison) else 0.0
    ),
    "expected_geometric_prediction_rows": int(expected_prediction_rows),
    "actual_final_geometric_prediction_rows": int(len(geometric_preds)),
    "protocol_summary_rows": int(len(protocol_summary)),
    "protocol_comparison_rows": int(len(comparison)),
    "residual_moran_rows": int(len(moran_summary)),
    "residual_fold_rows": int(len(residual_fold_summary)),
    "moran_knn_k": 8,
    "moran_permutations": 199,
    "n_positive_moran_i": int(moran_summary["moran_i"].gt(0).sum()),
    "n_moran_permutation_p_le_0_05": int(
        moran_summary["permutation_p_two_sided"].le(0.05).sum()
    ),
    "n_models_with_unresolved_repeated_alpha_edges": int(len(unresolved_edges)),
    "n_alpha_grid_edge_hits": int(
        final_edge_counts["edge_hits"].sum()
    ),
    "n_models_with_repeated_edge_hits": int(len(unresolved_edges)),
    "protocol_inference": (
        "descriptive; random, borough and geometric fold IDs are not paired"
    ),
    "residual_inference": (
        "held-out Moran's I is a diagnostic effect size; permutation p-values "
        "are not model-selection criteria"
    ),
    "integrity_gate_pass": integrity_gate_pass,
    "interpretation_gate_pass": interpretation_gate_pass,
}

atomic_csv(
    geometric_results.sort_values(
        ["task", "model_id", "geometric_outer_fold"]
    ),
    FINAL_REFINED_RESULTS_PATH,
)
atomic_json(audit, GEOMETRIC_AUDIT_PATH)

assert integrity_gate_pass, "Notebook 11 output completeness check failed"
if not interpretation_gate_pass:
    display(unresolved_edges)
    raise RuntimeError(
        "The targeted alpha refinement found a repeated unresolved upper "
        "boundary. Inspect the displayed models before interpretation."
    )

atomic_parquet(
    geometric_preds.sort_values(["task", "model_id", "sample_id"]),
    GEOMETRIC_CV_PREDICTIONS_PATH,
)
atomic_csv(
    protocol_summary.sort_values(
        ["task", "model_id", "validation_scheme"]
    ),
    GEOMETRIC_PROTOCOL_SUMMARY_PATH,
)
atomic_csv(
    comparison.sort_values(["task", "model_id"]),
    GEOMETRIC_PROTOCOL_COMPARISON_PATH,
)
atomic_csv(moran_summary, GEOMETRIC_RESIDUAL_MORAN_PATH)
atomic_csv(residual_fold_summary, GEOMETRIC_RESIDUAL_FOLD_PATH)

if len(refinement_comparison):
    display(refinement_comparison[[
        "task", "model_id", "geometric_outer_fold",
        "best_alpha_original", "best_alpha_refined",
        "delta_r2_refined_minus_original",
        "delta_rmse_refined_minus_original",
        "delta_mae_refined_minus_original",
    ]])

display(comparison[[
    "task", "model_id", "analysis_role",
    "pooled_r2_random", "pooled_r2_borough", "pooled_r2_geometric",
    "random_minus_borough_pooled_r2",
    "borough_minus_geometric_pooled_r2",
    "pooled_rmse_random", "pooled_rmse_borough", "pooled_rmse_geometric",
]])
display(moran_summary)
print(json.dumps(audit, indent=2))
print("Notebook 11 completed: final tables and residual diagnostics saved.")

## Interpretation

The three evaluation designs represent progressively harder forms of generalisation. Random division tests interpolation among mixed London observations; borough hold-out removes administrative groups; continuous-region hold-out removes an entire compact part of the city.

The largest change occurs for PTAL. A smooth location trend does not transfer reliably across a broad unseen area, while visual and environmental representations preserve considerably more predictive information. Full fusion remains the strongest model by mean R², although its pooled performance is almost identical to Street View content plus coverage information. This near tie should be described as comparable performance rather than a decisive victory for either model.

EPC is less sensitive to the geographical division. DINOv2 and the compact/full combinations remain useful beyond compact property controls, but direct construction-age and floor-area information produces the most stable performance. TESSERA contributes only a very small increment to this richer baseline, while adding all representations still reduces performance. The targeted penalty refinement confirms that this result is not an artefact of the original tuning-grid boundary.

Moran's I shows that neighbouring locations still tend to be over- or under-predicted together, particularly for PTAL. Representations reduce this pattern substantially relative to the control baselines, but do not remove it. This is evidence of remaining spatial structure in the prediction errors, not a causal result or a criterion for choosing a winning model. Because 199 permutations give a minimum attainable p-value of 0.005, the dissertation should emphasise the size and ordering of Moran's I rather than treating the identical p-values as separate discoveries.